# BA283 State-Space Training — S4D / S6 vs LSTM (Colab T4)

Trains **state-space models** (S4D, S6/Mamba) and the shipped **LSTM** baseline on the same
gain-conditioned SPICE dataset, then scores them apples-to-apples on **held-out-gain ESR**.
2025 comparative work (EURASIP JASMP) finds SSMs more accurate and faster-converging than LSTM
for virtual-analog modelling; this notebook measures whether that holds for the BA283.

Models live in the `train/` package behind `--model {lstm,s4d,s6}` (pure-PyTorch, recurrent,
state-threaded — no `mamba-ssm`/CUDA-kernel deps, runs on any runtime). Usability tweaks are
carried over from `ba283_redesign_colab_selective_scoring.ipynb`: a **config selector** and a
**checkpoint-recovery path**. The training cell also gates **GPU hyperparameter optimization**
behind a bool form field.

> Note: the recurrent SSM scan is a Python loop over the window — slower than cuDNN LSTM. Use a
> modest `WINDOW` and `HPO_EPOCHS`. Export/Rust inference is out of scope (LSTM-only schema).


## 1. Install rclone + upload `rclone.conf`


In [ ]:
import os, subprocess, shutil
from google.colab import files

if shutil.which('rclone') is None:
    subprocess.run('curl -s https://rclone.org/install.sh | sudo bash', shell=True, check=True)

if not os.path.exists('/root/.config/rclone/rclone.conf'):
    os.makedirs('/root/.config/rclone', exist_ok=True)
    up = files.upload()  # upload your rclone.conf
    name = next(iter(up))
    shutil.copy(name, '/root/.config/rclone/rclone.conf')

# First configured remote, e.g. 'gdrive' -> REMOTE = 'gdrive:'
remotes = subprocess.run(['rclone', 'listremotes'], capture_output=True, text=True).stdout.split()
REMOTE = remotes[0] if remotes else 'gdrive:'
print('REMOTE =', REMOTE)


## 2. Clone the repo + deps + GPU check


In [ ]:
import os, subprocess, torch
from getpass import getpass

REPO = 'ba283-vst3'
if not os.path.isdir(REPO):
    tok = getpass('GitHub token: ')
    url = f'https://{tok}@github.com/gustavokch/ba283-vst3.git'
    subprocess.run(['git', 'clone', url, REPO], check=True)
os.chdir('/content/' + REPO)
subprocess.run(['git', 'pull'], check=False)

subprocess.run(['pip', '-q', 'install', 'soundfile', 'pandas', 'tensorboard', 'scipy'], check=True)
# Pure-PyTorch SSMs need no extra dep; torch ships with Colab.

assert torch.cuda.is_available(), 'No CUDA — enable a GPU runtime (Runtime > Change runtime type).'
print('GPU:', torch.cuda.get_device_name(0))


## 3. Config — the LSTM baseline + S4D / S6 stack


In [ ]:
SEED, EPOCHS, PATIENCE = 745603, 60, 15
TARGET = 0.01  # ~ -20 dB redesign goal

FULL    = REMOTE + 'ba283_ablation/full9pt'   # 9-pt gain set (train 5-pt, val 9-pt)
RESULTS = REMOTE + 'ba283_ssm/results'         # verdict save path
BACKUP  = REMOTE + 'ba283_ssm'                 # checkpoint safety net

GAINS = {0.0, 0.25, 0.5, 0.75, 1.0}            # trained gains (held-out: the 4 between)

_STK = ['--stft-weight', '0.5', '--head', 'mlp', '--film', 'mlp']
_REG = ['--weight-decay', '1e-5', '--clip-norm', '1.0', '--lr', '1e-3']

CONFIGS = [
    dict(key='LSTM_REF', label='LSTM baseline',
         flags=['--model','lstm','--hidden','64','--layers','2','--dropout','0.1',*_STK,*_REG],
         eval=dict(model_type='lstm', hidden=64, head='mlp', film_kind='mlp', num_layers=2)),
    dict(key='S4D_0', label='S4D base',
         flags=['--model','s4d','--hidden','48','--layers','2','--d-state','64',*_REG],
         eval=dict(model_type='s4d', hidden=48, num_layers=2, d_state=64)),
    dict(key='S4D_1', label='S4D +stft+mlp',
         flags=['--model','s4d','--hidden','48','--layers','2','--d-state','64',*_STK,*_REG],
         eval=dict(model_type='s4d', hidden=48, num_layers=2, d_state=64, head='mlp', film_kind='mlp')),
    dict(key='S6_0', label='S6 base',
         flags=['--model','s6','--hidden','32','--layers','2','--d-state','16',*_REG],
         eval=dict(model_type='s6', hidden=32, num_layers=2, d_state=16)),
    dict(key='S6_1', label='S6 +stft+mlp',
         flags=['--model','s6','--hidden','32','--layers','2','--d-state','16',*_STK,*_REG],
         eval=dict(model_type='s6', hidden=32, num_layers=2, d_state=16, head='mlp', film_kind='mlp')),
]
print('configs:', [c['key'] for c in CONFIGS])


## 4. Stage the gain set (once) + remap manifests


In [ ]:
from train.stage import assert_matched_sr, remap_manifests, stage_rclone

DEST = '/content/full'
stage_rclone(FULL, DEST)
MAN = remap_manifests(DEST, names=('train', 'val'), out_dir=DEST)
sr = assert_matched_sr(MAN['train'], expected_sr=44100)
print('manifests:', MAN, 'sr:', sr)


## 5. Train — LSTM + S4D + S6, with optional GPU HPO

`SELECTED_CONFIG_KEYS_STR` filters which configs to train (`ALL` = whole stack).
`RUN_HPO` gates a GPU-only random hyperparameter search over `HPO_MODEL`; the best trial is then
trained to full `EPOCHS`. Leave it off for a plain stack run.


In [ ]:
SELECTED_CONFIG_KEYS_STR = "ALL"  # @param {type:"string", placeholder:"ALL"}
RUN_HPO = False  # @param {type:"boolean"}
HPO_MODEL = "s6"  # @param ["s4d", "s6"]
HPO_TRIALS = 12  # @param {type:"integer"}
HPO_EPOCHS = 20  # @param {type:"integer"}
WINDOW = 2048  # @param {type:"integer"}

import json, os, random, subprocess
import torch
from eval.ablation.run_checkpoint_eval import run_cell

COMMON = ['--train-manifest', MAN['train'], '--val-manifest', MAN['val'],
          '--device', 'cuda', '--seed', str(SEED), '--patience', str(PATIENCE),
          '--window', str(WINDOW), '--conditioning', 'film']

def backup_cfg(key):
    for folder in (f'ckpt_{key}', f'logs_{key}'):
        try:
            subprocess.run(['rclone', 'copy', folder, f"{BACKUP}/{folder}", '-P'], check=False)
        except Exception as exc:
            print('WARN backup failed', folder, exc)

def train_cfg(key, flags, epochs):
    ck = f'ckpt_{key}'
    if os.path.exists(f'{ck}/best.pth'):
        print('skip (done):', key); return
    cmd = ['python', '-m', 'train.train', *COMMON, *flags,
           '--epochs', str(epochs), '--checkpoint-dir', ck, '--log-dir', f'logs_{key}']
    print('train', key, ' '.join(flags))
    subprocess.run(cmd, check=True)
    backup_cfg(key)

if RUN_HPO:
    assert torch.cuda.is_available(), 'HPO is GPU-only.'
    rng = random.Random(SEED)
    space = dict(hidden=[32,48,64], layers=[1,2], lr=['3e-4','5e-4','1e-3'],
                 dropout=['0.0','0.1'], stft=['0.0','0.5'])
    space['d_state'] = [8,16,32] if HPO_MODEL == 's6' else [32,64,128]
    board = []
    for t in range(HPO_TRIALS):
        p = {k: rng.choice(v) for k, v in space.items()}
        key = f'HPO_{HPO_MODEL}_{t:02d}'
        flags = ['--model', HPO_MODEL, '--hidden', str(p['hidden']), '--layers', str(p['layers']),
                 '--dropout', str(p['dropout']), '--lr', str(p['lr']), '--d-state', str(p['d_state']),
                 '--stft-weight', str(p['stft']), '--head', 'mlp', '--film', 'mlp',
                 '--weight-decay', '1e-5', '--clip-norm', '1.0']
        train_cfg(key, flags, HPO_EPOCHS)
        ev = dict(model_type=HPO_MODEL, hidden=p['hidden'], num_layers=int(p['layers']),
                  d_state=p['d_state'], head='mlp', film_kind='mlp')
        r = run_cell(f'ckpt_{key}/best.pth', MAN['val'], cell=key, trained_gains=GAINS, **ev)
        score = r.get('held_out_gain_esr') or r['val_esr']
        board.append(dict(key=key, params=p, held_out=score, val_esr=r['val_esr']))
        print(f'trial {t}: held_out={score:.5f}  {p}')
    board.sort(key=lambda b: b['held_out'])
    os.makedirs('hpo_out', exist_ok=True)
    open('hpo_out/leaderboard.json', 'w').write(json.dumps(board, indent=2))
    md = '| rank | key | held-out | val_esr | params |\n|---|---|---|---|---|\n'
    for i, b in enumerate(board):
        md += f"| {i} | {b['key']} | {b['held_out']:.5f} | {b['val_esr']:.5f} | {b['params']} |\n"
    open('hpo_out/leaderboard.md', 'w').write(md)
    subprocess.run(['rclone', 'copy', 'hpo_out', f'{RESULTS}/hpo', '-P'], check=False)
    print(md)
    best = board[0]['params']
    best_flags = ['--model', HPO_MODEL, '--hidden', str(best['hidden']), '--layers', str(best['layers']),
                  '--dropout', str(best['dropout']), '--lr', str(best['lr']), '--d-state', str(best['d_state']),
                  '--stft-weight', str(best['stft']), '--head', 'mlp', '--film', 'mlp',
                  '--weight-decay', '1e-5', '--clip-norm', '1.0']
    train_cfg(f'{HPO_MODEL}_best', best_flags, EPOCHS)
    print('best config trained to full epochs as', f'{HPO_MODEL}_best')
else:
    keys = SELECTED_CONFIG_KEYS_STR.strip().upper()
    sel = [c['key'] for c in CONFIGS] if keys in ('ALL', '') else \
          [k.strip() for k in SELECTED_CONFIG_KEYS_STR.split(',') if k.strip()]
    for cfg in [c for c in CONFIGS if c['key'] in sel]:
        train_cfg(cfg['key'], cfg['flags'], EPOCHS)


## 6. Score the stack + verdict table (vs SPICE + ACME) + save to Drive

`EXISTING_PATH_RCLONE`: optional remote subpath to pull pre-trained `ckpt_*`/`logs_*` before
scoring (train on one runtime, score on another). `MATCH_TOL_DB`: how close to the ACME floor
counts as a match.

The dataset wet target **is** rendered SPICE, so every model's `val_esr` is already an
error-vs-SPICE — that is the validation against SPICE. The verdict floor is the **ACME.jl
physical model's own ESR-vs-SPICE** (read from `eval/acme_vs_spice/out/summary.json`, median
≈ −31.6 dB; constant fallback if absent), the best fidelity the circuit admits. The LSTM is now
just another scored candidate, not the reference.

The **✅ matches** column = the model lands within `MATCH_TOL_DB` of the ACME floor
(ΔdB ≤ tol). Negative ΔdB = tighter to SPICE than the analytical model.

> ACME's ESR is raw-volts + onset-skipped (`eval/acme_vs_spice`); the SSM `val_esr` is
> normalized-domain + warmup-trimmed. Both are "fraction of SPICE energy unexplained" —
> comparable as a reference line, not bit-identical metrics.


In [ ]:
EXISTING_PATH_RCLONE = ""  # @param {type:"string"}
MATCH_TOL_DB = 3.0  # @param {type:"number"}  "matches ACME" = val_esr within this many dB of the floor

import json, math, os, subprocess
from eval.ablation.run_checkpoint_eval import run_cell
from eval.ablation.report import delta_db

# --- Reference fidelity floor: how well the ACME physical model tracks SPICE ---
# The dataset wet target IS rendered SPICE, so every model's val_esr below is
# already an error-vs-SPICE. The baseline is therefore no longer the LSTM run but
# the ACME.jl DK-method model's own ESR-vs-SPICE -- the best physical-model
# fidelity the circuit admits. A model "matches ACME" when its val_esr is within
# MATCH_TOL_DB of the floor (ΔdB <= tol); negative ΔdB = tighter than ACME.
ACME_SUMMARY = 'eval/acme_vs_spice/out/summary.json'
ACME_REF_ESR_DB = -31.6  # fallback: committed acme_vs_spice median (4-pt gain, pink+sweep)
try:
    _med = json.load(open(ACME_SUMMARY))['esr_summary_db']['median']
    if _med is not None:
        ACME_REF_ESR_DB = float(_med)
        print('ACME reference read from', ACME_SUMMARY)
except (OSError, KeyError, TypeError, ValueError) as exc:
    print('ACME summary unavailable, using fallback:', exc)
ACME_REF_ESR = 10.0 ** (ACME_REF_ESR_DB / 10.0)  # dB -> linear ESR (power ratio)
print(f'ACME-vs-SPICE floor: {ACME_REF_ESR_DB:+.1f} dB  (linear ESR {ACME_REF_ESR:.5f})')
print(f'match tolerance: ΔdB <= {MATCH_TOL_DB:+.1f} dB')


def render_vs_acme(cells, floor_esr, tol_db):
    """Verdict table against the ACME floor. ✅ = val_esr within tol_db of floor."""
    hdr = ("| cell | val_esr | acme_floor | ΔdB | held-out-gain | matches | notes |\n"
           "|---|---|---|---|---|---|---|")
    out = [hdr]
    for c in cells:
        val = float(c['val_esr'])
        d = delta_db(val, floor_esr)
        d_str = '-inf' if math.isinf(d) else f'{d:+.1f}'
        hog = c.get('held_out_gain_esr')
        hog_str = '—' if hog is None else f'{float(hog):.4f}'
        matches = '✅' if d <= tol_db else '—'
        out.append(f"| {c['cell']} | {val:.4f} | {floor_esr:.4f} | {d_str} | "
                   f"{hog_str} | {matches} | {c.get('notes', '')} |")
    return '\n'.join(out)


if EXISTING_PATH_RCLONE.strip():
    root = BACKUP if not EXISTING_PATH_RCLONE.strip('/') else REMOTE + EXISTING_PATH_RCLONE.strip('/')
    for cfg in CONFIGS:
        for folder in (f"ckpt_{cfg['key']}", f"logs_{cfg['key']}"):
            if not os.path.exists(folder):
                subprocess.run(['rclone', 'copy', f'{root}/{folder}', folder, '-P'], check=False)

results = []
for cfg in CONFIGS:
    ck = f"ckpt_{cfg['key']}/best.pth"
    if not os.path.exists(ck):
        print('no checkpoint, skip:', cfg['key']); continue
    r = run_cell(ck, MAN['val'], cell=cfg['key'], trained_gains=GAINS, **cfg['eval'])
    r['notes'] = cfg['label']
    r['baseline_esr'] = ACME_REF_ESR  # measured against the ACME physical-model floor
    r['matches_acme'] = delta_db(float(r['val_esr']), ACME_REF_ESR) <= MATCH_TOL_DB
    results.append(r)

# Reference rows so the table reads explicitly against SPICE (the target, perfect)
# and ACME (the physical-model floor) rather than against any neural baseline.
ref_rows = [
    dict(cell='SPICE_REF', val_esr=0.0, baseline_esr=ACME_REF_ESR,
         held_out_gain_esr=None, notes='ground-truth target (dataset wet)'),
    dict(cell='ACME_REF', val_esr=ACME_REF_ESR, baseline_esr=ACME_REF_ESR,
         held_out_gain_esr=None, notes='DK-method physical model vs SPICE (raw-volts)'),
]

table = render_vs_acme(ref_rows + results, ACME_REF_ESR, MATCH_TOL_DB)
print(table)
print(f'\nFloor = ACME-vs-SPICE; each val_esr is that model vs the SPICE target.')
print(f'✅ matches = within {MATCH_TOL_DB:+.1f} dB of ACME (ΔdB < 0 = tighter than ACME).')
print('Select on HELD-OUT-GAIN esr (gain interpolation); val_esr is stem-overlap optimistic.')

os.makedirs('ssm_save', exist_ok=True)
payload = {'acme_ref_esr_db': ACME_REF_ESR_DB, 'acme_ref_esr': ACME_REF_ESR,
           'match_tol_db': MATCH_TOL_DB, 'results': results}
open('ssm_save/ssm_results.json', 'w').write(json.dumps(payload, indent=2, default=float))
open('ssm_save/ssm_table.md', 'w').write(table)
subprocess.run(['rclone', 'copy', 'ssm_save', RESULTS, '-P'], check=False)


## Next steps

If an SSM beats `LSTM_REF` on held-out-gain ESR, the follow-on (separate effort, not this
notebook) is a `ba283-ssm-v1` export schema + a Rust inference kernel for the plugin — the
recurrent scan here is already the streaming/sample-by-sample form that real-time inference needs.
